In [2]:
import pandas as pd
from pathlib import Path

airbnb = pd.read_csv("airbnb_filtered.csv", low_memory=False)
airbnb.head()

,listing_id,review_id,listing_name,description,review,rating
0,2992450,15066586,Luxury 2 bedroom apartment,The apartment is located in a quiet neighborho...,Large apartment; nice kitchen and bathroom. Ke...,3.56
1,2992450,21810844,Luxury 2 bedroom apartment,The apartment is located in a quiet neighborho...,"This may be a little late, but just to say Ken...",3.56
2,2992450,27434334,Luxury 2 bedroom apartment,The apartment is located in a quiet neighborho...,The apartment was very clean and convenient to...,3.56
3,2992450,28524578,Luxury 2 bedroom apartment,The apartment is located in a quiet neighborho...,Kenneth was ready when I got there and arrange...,3.56
4,2992450,35913434,Luxury 2 bedroom apartment,The apartment is located in a quiet neighborho...,We were pleased to see how 2nd Street and the ...,3.56


In [4]:
import numpy as np

def load_glove(path, dim):
    vectors = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.rstrip().split(" ")
            word = parts[0]
            vals = np.asarray(parts[1:], dtype=np.float32)
            if vals.shape[0] != dim:
                continue
            vectors[word] = vals
    return vectors

glove = load_glove("glove.6B/glove.6B.100d.txt", dim=100)


In [5]:
import re

token_re = re.compile(r"[a-z]+(?:'[a-z]+)?")

def text_to_glove_avg(text, glove, dim):
    
    tokens = token_re.findall(text.lower())
    vecs = [glove[t] for t in tokens if t in glove]

    if not vecs:
        return np.zeros(dim, dtype=np.float32)

    return np.mean(vecs, axis=0).astype(np.float32)


In [6]:
DIM = 100

airbnb["desc_emb"] = airbnb["description"].apply(lambda x: text_to_glove_avg(x, glove, DIM))
airbnb["review_emb"] = airbnb["review"].apply(lambda x: text_to_glove_avg(x, glove, DIM))
airbnb.head()


,listing_id,review_id,listing_name,description,review,rating,desc_emb,review_emb
0,2992450,15066586,Luxury 2 bedroom apartment,The apartment is located in a quiet neighborho...,Large apartment; nice kitchen and bathroom. Ke...,3.56,"[-0.047772855, 0.12813857, 0.23404115, -0.0663...","[-0.052746892, 0.16154227, 0.37204847, -0.2111..."
1,2992450,21810844,Luxury 2 bedroom apartment,The apartment is located in a quiet neighborho...,"This may be a little late, but just to say Ken...",3.56,"[-0.047772855, 0.12813857, 0.23404115, -0.0663...","[-0.1777499, 0.16362435, 0.34134158, -0.170131..."
2,2992450,27434334,Luxury 2 bedroom apartment,The apartment is located in a quiet neighborho...,The apartment was very clean and convenient to...,3.56,"[-0.047772855, 0.12813857, 0.23404115, -0.0663...","[-0.1447165, 0.20639701, 0.3322839, -0.2451596..."
3,2992450,28524578,Luxury 2 bedroom apartment,The apartment is located in a quiet neighborho...,Kenneth was ready when I got there and arrange...,3.56,"[-0.047772855, 0.12813857, 0.23404115, -0.0663...","[-0.04268333, 0.11577168, 0.35954562, -0.23441..."
4,2992450,35913434,Luxury 2 bedroom apartment,The apartment is located in a quiet neighborho...,We were pleased to see how 2nd Street and the ...,3.56,"[-0.047772855, 0.12813857, 0.23404115, -0.0663...","[-0.09856582, 0.16494532, 0.3176583, -0.207339..."


In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Fit TF-IDF on BOTH fields so weights are meaningful
corpus = pd.concat([airbnb["description"].fillna(""), airbnb["review"].fillna("")], axis=0).astype(str)

tfidf = TfidfVectorizer(lowercase=True, token_pattern=r"[a-z]+(?:'[a-z]+)?", min_df=2)
tfidf.fit(corpus)

vocab = tfidf.vocabulary_
idf = tfidf.idf_

def text_to_glove_tfidf(text, glove, dim, vocab, idf):

    tokens = token_re.findall(text.lower())
    num = np.zeros(dim, dtype=np.float32)
    den = 0.0

    for t in tokens:
        if t in glove and t in vocab:
            w = float(idf[vocab[t]])  # IDF weight
            num += w * glove[t]
            den += w

    if den == 0.0:
        return np.zeros(dim, dtype=np.float32)
    return (num / den).astype(np.float32)


In [8]:
DIM = 100

airbnb["desc_emb_tfidf"] = airbnb["description"].apply(lambda x: text_to_glove_tfidf(x, glove, DIM, vocab, idf))
airbnb["review_emb_tfidf"] = airbnb["review"].apply(lambda x: text_to_glove_tfidf(x, glove, DIM, vocab, idf))
airbnb.head()

,listing_id,review_id,listing_name,description,review,rating,desc_emb,review_emb,desc_emb_tfidf,review_emb_tfidf
0,2992450,15066586,Luxury 2 bedroom apartment,The apartment is located in a quiet neighborho...,Large apartment; nice kitchen and bathroom. Ke...,3.56,"[-0.047772855, 0.12813857, 0.23404115, -0.0663...","[-0.052746892, 0.16154227, 0.37204847, -0.2111...","[-0.07101469, 0.2064287, 0.16530348, 0.0034542...","[-0.024390519, 0.1772875, 0.33225054, -0.18312..."
1,2992450,21810844,Luxury 2 bedroom apartment,The apartment is located in a quiet neighborho...,"This may be a little late, but just to say Ken...",3.56,"[-0.047772855, 0.12813857, 0.23404115, -0.0663...","[-0.1777499, 0.16362435, 0.34134158, -0.170131...","[-0.07101469, 0.2064287, 0.16530348, 0.0034542...","[-0.16761875, 0.17117491, 0.27614823, -0.09748..."
2,2992450,27434334,Luxury 2 bedroom apartment,The apartment is located in a quiet neighborho...,The apartment was very clean and convenient to...,3.56,"[-0.047772855, 0.12813857, 0.23404115, -0.0663...","[-0.1447165, 0.20639701, 0.3322839, -0.2451596...","[-0.07101469, 0.2064287, 0.16530348, 0.0034542...","[-0.13655844, 0.24713281, 0.28488106, -0.23691..."
3,2992450,28524578,Luxury 2 bedroom apartment,The apartment is located in a quiet neighborho...,Kenneth was ready when I got there and arrange...,3.56,"[-0.047772855, 0.12813857, 0.23404115, -0.0663...","[-0.04268333, 0.11577168, 0.35954562, -0.23441...","[-0.07101469, 0.2064287, 0.16530348, 0.0034542...","[-0.05733155, 0.1252569, 0.3221885, -0.2068669..."
4,2992450,35913434,Luxury 2 bedroom apartment,The apartment is located in a quiet neighborho...,We were pleased to see how 2nd Street and the ...,3.56,"[-0.047772855, 0.12813857, 0.23404115, -0.0663...","[-0.09856582, 0.16494532, 0.3176583, -0.207339...","[-0.07101469, 0.2064287, 0.16530348, 0.0034542...","[-0.08970667, 0.19444233, 0.26255697, -0.17219..."


In [9]:
airbnb.to_csv("airbnb_glove_embeddings.csv", index=False)